# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

In [1]:
# Path assumes this notebook lives in work/notebooks/ — adjust if you moved it.
DATA_PATH = "../../data/raw/content_refresh_anonymized.csv"

import numpy as np
import pandas as pd

df = pd.read_csv(DATA_PATH)

# --- Fix informative-null / placeholder cases (see w03_data_contract) ---
df.loc[df["trend_direction"].isin(["flat", "new"]), "trend_pct"] = 0.0

# avg_position == 0 is a placeholder for "no real position data", not rank 0
has_real_position = (df["avg_position"] > 0).astype(int)
df.loc[df["avg_position"] == 0, "avg_position"] = np.nan

numeric_features = [
    "word_count", "char_count", "impressions_90d", "clicks_90d", "pageviews_90d",
    "sessions_90d", "users_90d", "engaged_sessions_90d", "ai_sessions_90d",
    "scroll_events_90d", "content_age_days", "days_since_last_update", "ctr",
    "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct", "trend_pct",
]
categorical_features = ["trend_direction"]

X_num = df[numeric_features].copy()
skewed_cols = [
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d",
    "users_90d", "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
]
for col in skewed_cols:
    X_num[col] = np.log1p(X_num[col])

word_count_was_missing = df["word_count"].isnull().astype(int)
char_count_was_missing = df["char_count"].isnull().astype(int)

X_num = X_num.fillna(X_num.median(numeric_only=True))
X_num["word_count_was_missing"] = word_count_was_missing
X_num["char_count_was_missing"] = char_count_was_missing
X_num["has_real_position"] = has_real_position

X_cat = pd.get_dummies(df[categorical_features].fillna("unknown"), drop_first=True)
X = pd.concat([X_num, X_cat], axis=1).fillna(0)
print("Feature matrix shape:", X.shape)
print(X.columns.tolist())

Feature matrix shape: (30000, 25)
['word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'trend_pct', 'word_count_was_missing', 'char_count_was_missing', 'has_real_position', 'trend_direction_flat', 'trend_direction_new', 'trend_direction_stable', 'trend_direction_up']


## 2. Feature notes (meaning, missing, categorical, available-when?)

| Feature | Meaning | Missing handling | Available before decision point? |
|---|---|---|---|
| `impressions_90d`, `clicks_90d`, `pageviews_90d`, `sessions_90d`, `users_90d` | Raw traffic counts, log-transformed (heavy right tail) | No nulls in this data | Yes — observed history |
| `engaged_sessions_90d`, `ai_sessions_90d`, `scroll_events_90d` | Engagement/AI-referral volume | No nulls | Yes |
| `word_count`, `char_count` | Content depth | ~26% null, concentrated in `keyword article`; median-imputed + kept a `_was_missing` flag rather than dropped rows | Yes, when tracked |
| `avg_position` | Average SERP position | 0 is a placeholder for "no real position data" (near-zero impressions) — converted to NaN + `has_real_position` flag, then median-imputed | Yes, when meaningful |
| `trend_pct` | % change vs prior period | Null exactly when `trend_direction` is 'flat'/'new' — filled with 0 (a real, not missing, value) | Yes |
| `ctr`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct` | Derived rates | No nulls | Yes |
| `content_age_days`, `days_since_last_update` | Content lifecycle | No nulls | Yes |
| `trend_direction` (categorical, one-hot) | Directional bucket | Never null after the trend_pct fix | Yes |

## 3. The leakage hunt

There's no supervised label here, so classic label-leakage doesn't apply the same way it would for a scoring/prediction lane. Still worth attacking:

In [2]:
# Check: none of FlyRank's product-decision fields are in this dataset at all
product_flags = ["health_score", "priority_score", "action_type", "needs_ctr_fix", "is_quick_win"]
present = [c for c in product_flags if c in df.columns]
print("Product decision flags present in data (should be empty):", present)

# Check: none of the *_tier columns (redundant bucketed versions) made it into X
tier_cols = [c for c in df.columns if "tier" in c]
leaked_tiers = [c for c in tier_cols if c in X.columns]
print("Tier columns present in feature matrix (should be empty):", leaked_tiers)

Product decision flags present in data (should be empty): []
Tier columns present in feature matrix (should be empty): []


## 4. What I excluded and why

- `search_volume`, `competition`, `cpc` — structurally 0/null for non-keyword content; including them just re-encodes `content_type`.
- `age_tier`, `age_tier_order`, `freshness_tier`, `word_count_tier`, `char_count_tier`, `impression_tier`, `position_tier` — redundant bucketed versions of numeric columns already used raw.
- `provider_used`, `model_used` — content-generation metadata, not a performance signal.
- `content_type`, `main_intent`, `competition_level` — tested as clustering inputs and dropped after inspection: their missing-value dummy category ("unknown") perfectly separated `feedly article` content, so clusters were just rediscovering content_type rather than finding a behavioral pattern. Kept for **profiling** clusters afterward instead.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words (observed / directional / decision-support), never causal or 'predicting Google'